In [38]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

***CARGA DE DATOS***

In [49]:
data_folder = r"C:\Users\erik-\OneDrive\Documentos\GitHub\DETECCION-DE-ANOMALIAS\data"

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv"
]

***EXPLORACIÓN DE DATOS Y COLUMNAS***

In [53]:
selected_features = [
    'Flow Duration',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Packet Length Mean',
    'Label'
]

***PREPROCESAMIENTO***

In [55]:
def preprocess_cicids(file_path, start_date):

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # asegurar columnas
    df = df[selected_features].copy()

    # limpiar infinitos
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    # timestamp simulado
    df['time'] = pd.date_range(
        start=pd.to_datetime(start_date),
        periods=len(df),
        freq='s'
    )

    df.set_index('time', inplace=True)

    feature_cols = [
        'Flow Duration',
        'Flow Bytes/s',
        'Flow Packets/s',
        'Total Fwd Packets',
        'Total Backward Packets',
        'Total Length of Fwd Packets',
        'Total Length of Bwd Packets',
        'Fwd Packets/s',
        'Bwd Packets/s',
        'Packet Length Mean'
    ]

    ts_features = df.resample('1s')[feature_cols].agg({
        'Flow Duration': 'mean',
        'Flow Bytes/s': 'mean',
        'Flow Packets/s': 'mean',
        'Total Fwd Packets': 'sum',
        'Total Backward Packets': 'sum',
        'Total Length of Fwd Packets': 'sum',
        'Total Length of Bwd Packets': 'sum',
        'Fwd Packets/s': 'mean',
        'Bwd Packets/s': 'mean',
        'Packet Length Mean': 'mean'
    })

    ts_features['flow_count'] = df.resample('1s').size()

    ts_label = df['Label'].resample('1s').apply(
        lambda x: 'ATTACK' if (x != 'BENIGN').any() else 'BENIGN'
    )

    ts = ts_features.copy()
    ts['Label'] = ts_label

    ts = ts.fillna(0)

    return ts

In [56]:
processed_folder = r"C:\Users\erik-\OneDrive\Documentos\GitHub\DETECCION-DE-ANOMALIAS\processed_data"
os.makedirs(processed_folder, exist_ok=True)

processed = {}

for file in files:
    path = os.path.join(data_folder, file)

    ts = preprocess_cicids(path, start_dates[file])

    processed[file] = ts

    # guardar cada archivo ya procesado
    out_path = os.path.join(processed_folder, file.replace(".csv", "_processed.csv"))
    ts.to_csv(out_path)

print("Procesados y guardados:", len(processed))

Procesados y guardados: 8
